# Design of Weldments - Implementation Notes

Original Python implementations of section property calculations,
including `SectionComponent` and `Section` classes for built-up
cross-section analysis. Concepts reference:

- Blodgett, O.W. *Design of Weldments*. Lincoln Electric.

Refer to the original text for theoretical background and derivations.


# 1. Design Approach

## 1.1 Progress Through Welded Steel Construction

Provides history and background of welding process developement. The goal of weld design is to produce the maximum performance for the least cost. Outlines the benefits of designing and implementing design solutions with welded steel.

## 1.2 Systemic Design of Weldments

### 1.2.7 The Member Factor in Design Formulas

## 1.3 Problem Definition

In [6]:
from civilpy.general import units

In [15]:
t_1 = 1 * units.inch
t_2 = .5 * units.inch

A_1 = 38.0 * units.inch ** 2
A_2 = 19.5 * units.inch ** 2

I_1 = 359.7 * units.inch ** 4
I_2 = 193.8 * units.inch ** 4

S_1 = 50.5 * units.inch ** 3
S_2 = 23.9 * units.inch ** 3

print(f"t ratio = {(t_2/t_1 * 100).magnitude}%")
print(f"A ratio = {(round(A_2/A_1 * 100, 1)).magnitude}%")
print(f"I ratio = {(round(I_2/I_1 * 100, 1)).magnitude}%")  # Book has this value incorrect
print(f"S ratio = {(round(S_2/S_1 * 100, 1)).magnitude}%")

In [65]:
a_c = 0
b_c = .75 * units.inch

$$ S = \frac{I}{c} = \frac{1.34}{2} = 0.67 $$

In [71]:
I_a = (15 * units.inch * .75 * units.inch ** 3) / 12
I_b = (.25 * units.inch * .375 * units.inch ** 3) / 12

A_a = 15 * units.inch * .75 * units.inch
A_b = .25 * units.inch * .375 * units.inch

Ad_a_2 = 0
Ad_b_2 = A_b * b_c ** 2

I_a + Ad_a_2 + I_b + Ad_b_2

<b><u>Python Demo - Calculating Section properties of a member</u></b>

NOTE: For production use, prefer the <code>CrossSection</code> class from <code>civilpy.structural.section_properties</code>.<br>
Visualization of built-up sections can be added with matplotlib by drawing rectangles for each <code>SectionComponent</code>.

In [152]:
class SectionComponent:
    """
    Class to hold the values for the individual members of a built up section that will later
    be used to calculate properties of the full section like neutral axis, momement of inertia, etc.

    If the shape is anything other than a rectangle (representing a plate) the area of the shape must
    be passed to the object as a value and the `irregular_shape` value should be set to `True`.
    """
    def __init__(self, dimensions = (None, None), offset=(None, None), irregular_shape=False, area=None, I=None):
        self.x = dimensions[0]
        self.y = dimensions[1]
        self.x_bar = offset[0] + (self.x / 2)
        self.y_bar = offset[1] + (self.y / 2)
        self.irregular_shape = irregular_shape
        self.area = area
        self.I_own = I  # Moment of inertia about own centroid (required for irregular shapes)
        self.calculate_component_properties()

    def calculate_component_properties(self):
        if not self.irregular_shape:
            self.area = self.x * self.y

In [165]:
test_object = SectionComponent(dimensions = (2 * units.inch, 10 * units.inch), offset = (0. * units.inch, 0. * units.inch))
print(test_object.x_bar)

test_object_2 = SectionComponent(dimensions = (20 * units.inch, 5 * units.inch), offset = (0. * units.inch, 10. * units.inch))

In [177]:
class Section:
    """
    Class to define a built up section and calculate the section properties of the 
    built up section.
    """
    def __init__(self, components=[]):
        self.components = components
        self.area = sum([component.area for component in self.components])
        self.neutral_y_axis = sum([component.area*component.y_bar for component in self.components]) / self.area
        self.neutral_x_axis = sum([component.area*component.x_bar for component in self.components]) / self.area
        self.moment_of_inertia_x()

    def moment_of_inertia_x(self):
        total_I = 0
        for component in self.components:
            if not component.irregular_shape:
                # Parallel axis theorem: I_own + A*d^2
                I_own = (component.x * component.y ** 3) / 12
                d = component.y_bar - self.neutral_y_axis
                total_I += I_own + d ** 2 * component.area
            else:
                # For irregular shapes, provide I via SectionComponent(I=...) argument
                if hasattr(component, "I_own") and component.I_own is not None:
                    d = component.y_bar - self.neutral_y_axis
                    total_I += component.I_own + d ** 2 * component.area
                # else: I_own not supplied; contribution skipped
        self.I_x = total_I


In [178]:
test_beam = Section(components=[test_object, test_object_2])
test_beam.I_x

Neutral axis calculation

$$ c = \frac{\sum_i \bar y_i  \bar A }{\sum_i \bar A } $$

In [179]:
# As a demonstration, here's a demonstration of an irregular section
# The section consists of a 40"x5" top flange and a 5"x35" web
c = (
       (37.5 * units.inch * 40 * units.inch * 5 * units.inch) + 
       (5 * units.inch * 35 * units.inch * 17.5 * units.inch)
    ) / (
        5 * units.inch * 35 * units.inch + 5 * units.inch * 40 * units.inch
    )

print(round(c, 3))

In [180]:
A_1 = 5 * units.inch * 35 * units.inch
A_2 = 40 * units.inch * 5 * units.inch

In [181]:
y_1 = c - 35 * units.inch / 2
y_2 = 35 * units.inch + 2.5 * units.inch - c

In [183]:
I_1 = (5 * units.inch * (35 * units.inch) ** 3) / 12
I_2 = (40 * units.inch * (5 * units.inch) ** 3) / 12

In [184]:
I = I_1 + y_1 ** 2 * A_1 + I_2 + y_2 ** 2 * A_2
round(I, 1)

An example of the process by integration

In [185]:
from sympy import *

In [186]:
y = Symbol('y')

# Define the two integrals
integral1 = integrate(5*y**2, (y, -28.167, 6.833))
integral2 = integrate(40*y**2, (y, 6.833, 11.833))

# Calculate the sum of the integrals
total_integral = integral1 + integral2
round(total_integral * units('in^4'), 1)

In [188]:
delta_al = (1 * (2.8/7.9) * 1 * 1 ** 3 ) / ((10.3e6 / 30e6) * 1)
round(delta_al, 2)

## 1.4 - Designers Guide to Effecient Use of Steel